# EEG_08 — GNN Classification: GCN · GAT · DANN

Trains and evaluates three GNN architectures for 4-class imagined-speech decoding.

| Model | Description |
|-------|-------------|
| GCN   | 3-layer GCNConv + global mean pool |
| GAT   | 3-layer GATConv multi-head + global mean pool |
| DANN  | GCN encoder + task head + domain head (GRL) |

**Input**: pre-built `.pt` graph files from `data/graphs_abs_pcc/`  
**Split**: subject-independent, trial-level global shuffle  
**Primary metric**: balanced accuracy (4-class imbalanced)

## 1 — Imports & Config

In [1]:
import json
import logging
import os
import random
import re
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import wandb
import weave
from sklearn.metrics import (
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
)
from torch.utils.data import Dataset as TorchDataset, WeightedRandomSampler
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as PyGDataLoader
from torch_geometric.nn import GATConv, GCNConv, global_mean_pool
from tqdm.auto import tqdm

# ── logging ──────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("eeg08")

# ── project root ──────────────────────────────────────────────────────────────
project_root = next(
    (p for p in [Path.cwd()] + list(Path.cwd().parents) if (p / ".git").exists()),
    Path.cwd(),
)
(project_root / "checkpoints").mkdir(exist_ok=True)
(project_root / "figures").mkdir(exist_ok=True)
log.info(f"project_root: {project_root}")

# ── config ────────────────────────────────────────────────────────────────────
CONFIG = {
    # data
    "data_root":       str(project_root / "data" / "graphs_abs_pcc"),
    "n_classes":       4,
    # split
    "train_ratio":     0.70,
    "val_ratio":       0.15,
    "test_ratio":      0.15,
    # training
    "batch_size":      32,
    "epochs":          100,
    "lr":              1e-3,
    "early_stopping":  20,
    "seed":            42,
    "device":          "cuda" if torch.cuda.is_available() else "cpu",
    # architecture
    "hidden_dim":      64,
    "gat_heads":       4,
    "dropout":         0.3,
    # DANN
    "dann_lambda":     1.0,
    # imbalance
    "class_weighting": "loss",     # "loss" | "sampler"
    # W&B
    "wandb_project":   "miralis-imagined-speech",
    "wandb_entity":    "uras-daniele22-politecnico-di-milano",
}

torch.manual_seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
log.info(f"device={CONFIG['device']}  seed={CONFIG['seed']}")

# helper
n_params = lambda m: sum(p.numel() for p in m.parameters() if p.requires_grad)

/home/daniele_u/miniconda3/envs/daniele_311/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(
01:29:51  INFO      project_root: /home/daniele_u/miralis-hypergraph-imagined-speech
01:29:51  INFO      device=cuda  seed=42


## 2 — W&B + Weave Setup

Set your API key **before** running this cell:
```bash
export WANDB_API_KEY=<your_key>
```
Never hardcode API keys in notebooks.

In [2]:
# Reads WANDB_API_KEY from environment automatically.
# If the variable is not set, wandb.login() will open an interactive prompt.
wandb.login()
weave.init(project_name=CONFIG["wandb_project"])
log.info("W&B and Weave initialised.")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/daniele_u/.netrc.
wandb: Currently logged in as: uras-daniele22 (uras-daniele22-politecnico-di-milano) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
01:29:51  INFO      HTTP Request: POST https://api.wandb.ai/graphql "HTTP/1.1 200 OK"
01:29:51  INFO      HTTP Request: GET https://trace.wandb.ai/server_info "HTTP/1.1 200 OK"
01:29:52  INFO      HTTP Request: POST https://api.wandb.ai/graphql "HTTP/1.1 200 OK"
01:29:52  INFO      HTTP Request: POST https://api.wandb.ai/graphql "HTTP/1.1 200 OK"
01:29:52  INFO      HTTP Request: GET https://trace.wandb.ai/server_info "HTTP/1.1 200 OK"
01:29:52  INFO      HTTP Request: GET https://pypi.org/pypi/wandb/json "HTTP/1.1 200 OK"
01:29:52  INFO      HTTP Request: GET https://pypi.org/pypi/weave/json "HTTP/1.1 200 OK"
weave: Logged in as Weights & Biases user: uras-daniele22.
weave: View Weave data at https://wandb.ai/uras-daniele22-politecnico-di

## 3 — Dataset Loading & Subject-Independent Splits

In [3]:
class EEGGraphDataset(TorchDataset):
    """
    Lazy-loading dataset for pre-built .pt EEG graph files.

    Expected .pt content
    --------------------
    edge_index  : LongTensor  [2, E]
    edge_attr   : FloatTensor [E] or [E, F]
    x           : FloatTensor [N_electrodes, N_features]
    y           : LongTensor  scalar — word label (0-109), mapped to cluster (0-3)
    adj         : FloatTensor [N, N]
    meta        : dict        must contain key "subject_id" (int)

    Parameters
    ----------
    label2cluster : LongTensor [110]  word_label → cluster_id mapping
    """

    def __init__(self, pt_paths: list, subject_id_map: dict,
                 label2cluster: torch.Tensor):
        self.paths          = list(pt_paths)
        self.subject_id_map = subject_id_map
        self.label2cluster  = label2cluster   # [110] LongTensor

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int) -> Data:
        d  = torch.load(self.paths[idx], weights_only=False)

        # ── edge features ────────────────────────────────────────────
        ea = d["edge_attr"]
        if not isinstance(ea, torch.Tensor):
            ea = torch.tensor(np.asarray(ea), dtype=torch.float32)
        ea = ea.float()
        ew    = ea if ea.dim() == 1 else ea[:, 0]   # scalar weight for GCNConv
        ea_2d = ea.unsqueeze(1) if ea.dim() == 1 else ea

        # ── label: word (0-109) → cluster (0-3) ──────────────────────
        y_raw  = d["y"]
        y_word = int(y_raw.squeeze()) if isinstance(y_raw, torch.Tensor) else int(y_raw)
        y      = self.label2cluster[y_word]          # LongTensor scalar

        # ── subject ID → consecutive index ────────────────────────────
        raw_sid    = int(d["meta"]["subject_id"])
        mapped_sid = self.subject_id_map.get(raw_sid, 0)

        return Data(
            x           = d["x"].float(),
            edge_index  = d["edge_index"].long(),
            edge_attr   = ea_2d,
            edge_weight = ew,
            y           = y,
            subject_id  = torch.tensor(mapped_sid, dtype=torch.long),
        )


In [4]:
_PAT = re.compile(r"^P(\d+)_S(\d+)$")

def _subj_from_path(p: Path) -> int:
    """Parse subject ID from path parent folder: P{XXX}_S{YYY} → XXX."""
    m = _PAT.match(p.parent.name)
    return int(m.group(1)) if m else -1

# ── word label → cluster mapping ─────────────────────────────────────────────
# y in .pt files = word label (0-109); task = 4-class cluster (0-3)
_mapping_path = project_root / "configs" / "label_schemes" / "labelid2cluster_concr4.json"
assert _mapping_path.exists(), f"Mapping not found: {_mapping_path}"
_raw_mapping  = json.loads(_mapping_path.read_text())
LABEL2CLUSTER = torch.zeros(110, dtype=torch.long)
for word_str, cluster_id in _raw_mapping.items():
    LABEL2CLUSTER[int(word_str)] = int(cluster_id)
log.info(f"label2cluster loaded: {len(_raw_mapping)} words → {CONFIG['n_classes']} clusters")
log.info(f"Cluster distribution: {torch.bincount(LABEL2CLUSTER).tolist()}")

# ── collect all .pt paths ────────────────────────────────────────────────────
_data_root = Path(CONFIG["data_root"])
assert _data_root.exists(), f"data_root not found: {_data_root}"

all_paths = sorted(_data_root.rglob("trial_*.pt"))
log.info(f"Total .pt files: {len(all_paths)}")
if not all_paths:
    raise FileNotFoundError(f"No trial_*.pt in {_data_root}")

# ── subject ID mapping: raw → consecutive int ────────────────────────────────
raw_subject_ids = sorted({_subj_from_path(p) for p in all_paths})
subject_id_map  = {raw: idx for idx, raw in enumerate(raw_subject_ids)}
N_SUBJECTS      = len(raw_subject_ids)
log.info(f"Subjects: {N_SUBJECTS}")

# ── detect input shape from first sample ────────────────────────────────────
_s          = torch.load(all_paths[0], weights_only=False)
IN_CHANNELS = int(_s["x"].shape[1])
EDGE_DIM    = 1 if _s["edge_attr"].dim() == 1 else int(_s["edge_attr"].shape[1])
log.info(f"in_channels={IN_CHANNELS}  edge_dim={EDGE_DIM}  n_subjects={N_SUBJECTS}")

# ── class names ──────────────────────────────────────────────────────────────
_names_path = project_root / "configs" / "label_schemes" / "cluster_names_concr4.json"
LABEL_NAMES = (
    [json.loads(_names_path.read_text())[str(i)] for i in range(CONFIG["n_classes"])]
    if _names_path.exists()
    else [f"class_{i}" for i in range(CONFIG["n_classes"])]
)
log.info(f"Labels: {LABEL_NAMES}")

# ── trial-level global shuffle + split ──────────────────────────────────────
rng      = random.Random(CONFIG["seed"])
shuffled = list(all_paths)
rng.shuffle(shuffled)

n_total     = len(shuffled)
n_train     = int(n_total * CONFIG["train_ratio"])
n_val       = int(n_total * CONFIG["val_ratio"])
train_paths = shuffled[:n_train]
val_paths   = shuffled[n_train : n_train + n_val]
test_paths  = shuffled[n_train + n_val :]
log.info(f"Split → train:{len(train_paths)}  val:{len(val_paths)}  test:{len(test_paths)}")

# ── class weights from TRAIN split only (on cluster labels, not word labels) ─
log.info("Scanning train labels for class weights …")
train_labels = []
for p in tqdm(train_paths, desc="train labels", leave=False):
    d      = torch.load(p, weights_only=False)
    y_raw  = d["y"]
    y_word = int(y_raw.squeeze()) if isinstance(y_raw, torch.Tensor) else int(y_raw)
    train_labels.append(int(LABEL2CLUSTER[y_word]))   # ← cluster id, not word id

train_labels  = np.array(train_labels)
class_counts  = np.bincount(train_labels, minlength=CONFIG["n_classes"]).astype(float)
class_weights = torch.tensor(
    len(train_labels) / (CONFIG["n_classes"] * class_counts),
    dtype=torch.float32,
)
log.info(f"Class counts (train): {class_counts.astype(int).tolist()}")
log.info(f"Class weights:        {class_weights.numpy().round(3).tolist()}")

# ── datasets ─────────────────────────────────────────────────────────────────
train_ds = EEGGraphDataset(train_paths, subject_id_map, LABEL2CLUSTER)
val_ds   = EEGGraphDataset(val_paths,   subject_id_map, LABEL2CLUSTER)
test_ds  = EEGGraphDataset(test_paths,  subject_id_map, LABEL2CLUSTER)

# ── data loaders ─────────────────────────────────────────────────────────────
if CONFIG["class_weighting"] == "sampler":
    _sample_w    = class_weights[torch.tensor(train_labels)]
    _sampler     = WeightedRandomSampler(_sample_w, len(_sample_w), replacement=True)
    train_loader = PyGDataLoader(train_ds, batch_size=CONFIG["batch_size"], sampler=_sampler)
    _loss_weights = None
else:
    train_loader  = PyGDataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True)
    _loss_weights = class_weights

val_loader  = PyGDataLoader(val_ds,  batch_size=CONFIG["batch_size"], shuffle=False)
test_loader = PyGDataLoader(test_ds, batch_size=CONFIG["batch_size"], shuffle=False)
log.info(f"Train batches: {len(train_loader)}")


01:29:52  INFO      label2cluster loaded: 110 words → 4 clusters
01:29:52  INFO      Cluster distribution: [18, 27, 21, 44]
01:29:52  INFO      Total .pt files: 38883
01:29:52  INFO      Subjects: 74
01:29:52  INFO      in_channels=384  edge_dim=1  n_subjects=74
01:29:52  INFO      Labels: ['CONCR', 'AZIONE', 'STATO', 'ASTRATTO']
01:29:52  INFO      Split → train:27218  val:5832  test:5833
01:29:52  INFO      Scanning train labels for class weights …


train labels:   0%|          | 0/27218 [00:00<?, ?it/s]

01:29:58  INFO      Class counts (train): [4463, 6713, 5139, 10903]
01:29:58  INFO      Class weights:        [1.524999976158142, 1.0140000581741333, 1.3240000009536743, 0.6240000128746033]
01:29:58  INFO      Train batches: 851


## 4 — Model Definitions

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# Gradient Reversal Layer
# ─────────────────────────────────────────────────────────────────────────────
class _GRLFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lam):
        ctx.lam = lam
        return x.clone()

    @staticmethod
    def backward(ctx, grad):
        return -ctx.lam * grad, None


class GradientReversal(nn.Module):
    def __init__(self, lam: float = 1.0):
        super().__init__()
        self.lam = lam

    def forward(self, x):
        return _GRLFunction.apply(x, self.lam)


# ─────────────────────────────────────────────────────────────────────────────
# Shared MLP head factory
# ─────────────────────────────────────────────────────────────────────────────
def _mlp_head(in_dim: int, out_dim: int, dropout: float) -> nn.Sequential:
    return nn.Sequential(
        nn.Linear(in_dim, in_dim // 2),
        nn.ReLU(),
        nn.Dropout(dropout),
        nn.Linear(in_dim // 2, out_dim),
    )


# ─────────────────────────────────────────────────────────────────────────────
# Model A — Baseline GCN
# ─────────────────────────────────────────────────────────────────────────────
class GCNModel(nn.Module):
    """3 × GCNConv (ReLU + dropout) → global mean pool → MLP → n_classes."""

    def __init__(self, in_channels, hidden_dim, n_classes, dropout=0.3):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_dim)
        self.conv2 = GCNConv(hidden_dim,  hidden_dim)
        self.conv3 = GCNConv(hidden_dim,  hidden_dim)
        self.drop  = nn.Dropout(dropout)
        self.head  = _mlp_head(hidden_dim, n_classes, dropout)

    def forward(self, data: Data) -> torch.Tensor:
        x, ei = data.x, data.edge_index
        ew     = getattr(data, "edge_weight", None)
        x = F.relu(self.conv1(x, ei, ew)); x = self.drop(x)
        x = F.relu(self.conv2(x, ei, ew)); x = self.drop(x)
        x = F.relu(self.conv3(x, ei, ew))
        return self.head(global_mean_pool(x, data.batch))


# ─────────────────────────────────────────────────────────────────────────────
# Model B — Graph Attention Network
# ─────────────────────────────────────────────────────────────────────────────
class GATModel(nn.Module):
    """
    3 × GATConv → global mean pool → MLP → n_classes.
    Layers 1-2: concat heads (dim × heads).
    Layer 3:    average heads (dim).
    """

    def __init__(self, in_channels, hidden_dim, n_classes, heads=4, dropout=0.3):
        super().__init__()
        self.conv1 = GATConv(in_channels,        hidden_dim, heads=heads, concat=True,  dropout=dropout)
        self.conv2 = GATConv(hidden_dim * heads,  hidden_dim, heads=heads, concat=True,  dropout=dropout)
        self.conv3 = GATConv(hidden_dim * heads,  hidden_dim, heads=1,     concat=False, dropout=dropout)
        self.drop  = nn.Dropout(dropout)
        self.head  = _mlp_head(hidden_dim, n_classes, dropout)

    def forward(self, data: Data) -> torch.Tensor:
        x, ei = data.x, data.edge_index
        x = F.elu(self.conv1(x, ei)); x = self.drop(x)
        x = F.elu(self.conv2(x, ei)); x = self.drop(x)
        x = F.elu(self.conv3(x, ei))
        return self.head(global_mean_pool(x, data.batch))


# ─────────────────────────────────────────────────────────────────────────────
# Model C — Domain-Adversarial Neural Network (DANN)
# ─────────────────────────────────────────────────────────────────────────────
class DANNModel(nn.Module):
    """
    Shared GCN encoder → two heads:
      • task head:   → n_classes logits
      • domain head: → n_subjects logits  (via GRL — gradients reversed)

    Training loss = task_loss + dann_lambda * domain_loss
    Inference:     only task head is used (predict method).
    """

    def __init__(self, in_channels, hidden_dim, n_classes, n_subjects,
                 dann_lambda=1.0, dropout=0.3):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_dim)
        self.conv2 = GCNConv(hidden_dim,  hidden_dim)
        self.conv3 = GCNConv(hidden_dim,  hidden_dim)
        self.drop  = nn.Dropout(dropout)

        self.task_head   = _mlp_head(hidden_dim, n_classes,  dropout)
        self.grl         = GradientReversal(lam=dann_lambda)
        self.domain_head = _mlp_head(hidden_dim, n_subjects, dropout)

    def _encode(self, data: Data) -> torch.Tensor:
        x, ei = data.x, data.edge_index
        ew     = getattr(data, "edge_weight", None)
        x = F.relu(self.conv1(x, ei, ew)); x = self.drop(x)
        x = F.relu(self.conv2(x, ei, ew)); x = self.drop(x)
        x = F.relu(self.conv3(x, ei, ew))
        return global_mean_pool(x, data.batch)   # [B, hidden_dim]

    def forward(self, data: Data):
        """Returns (task_logits, domain_logits) — training only."""
        z = self._encode(data)
        return self.task_head(z), self.domain_head(self.grl(z))

    def predict(self, data: Data) -> torch.Tensor:
        """Returns task_logits only — inference/evaluation."""
        return self.task_head(self._encode(data))


log.info("Model definitions OK.")

01:29:58  INFO      Model definitions OK.


## 5 — Training Loop

In [8]:
def train_model(
    model: nn.Module,
    train_loader,
    val_loader,
    config: dict,
    model_name: str,
    class_weights=None,
    is_dann: bool = False,
):
    """
    Unified training function for GCN, GAT, DANN.

    Parameters
    ----------
    class_weights : FloatTensor [n_classes] or None
    is_dann       : enables dual-head forward + domain loss

    Returns
    -------
    (trained_model, history_dict)
    """
    device    = torch.device(config["device"])
    model     = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"])
    criterion = nn.CrossEntropyLoss(
        weight=class_weights.to(device) if class_weights is not None else None
    )

    ckpt_path = project_root / "checkpoints" / f"eeg08_{model_name}_best.pt"

    run = wandb.init(
        project = config["wandb_project"],
        entity  = config.get("wandb_entity"),
        name    = model_name,
        config  = config,
        reinit  = "True",
    )

    best_val_bacc    = -1.0
    patience_counter = 0
    history = {"train_loss": [], "val_loss": [], "train_bacc": [], "val_bacc": []}
    t_start = time.time()

    for epoch in tqdm(range(config["epochs"]), desc=f"[{model_name}]", leave=True):

        # ── train ─────────────────────────────────────────────────────────────
        model.train()
        ep_loss, ep_preds, ep_labels = [], [], []

        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()

            if is_dann:
                task_logits, domain_logits = model(batch)
                loss = (
                    criterion(task_logits, batch.y)
                    + config["dann_lambda"] * nn.CrossEntropyLoss()(domain_logits, batch.subject_id)
                )
                logits = task_logits
            else:
                logits = model(batch)
                loss   = criterion(logits, batch.y)

            loss.backward()
            optimizer.step()

            ep_loss.append(loss.item())
            ep_preds.extend(logits.detach().argmax(1).cpu().numpy())
            ep_labels.extend(batch.y.cpu().numpy())

        # ── val ───────────────────────────────────────────────────────────────
        model.eval()
        val_loss, val_preds, val_labels = [], [], []

        with torch.no_grad():
            for batch in val_loader:
                batch  = batch.to(device)
                logits = model.predict(batch) if is_dann else model(batch)
                val_loss.append(criterion(logits, batch.y).item())
                val_preds.extend(logits.argmax(1).cpu().numpy())
                val_labels.extend(batch.y.cpu().numpy())

        # ── metrics ───────────────────────────────────────────────────────────
        tl   = float(np.mean(ep_loss))
        vl   = float(np.mean(val_loss))
        tba  = balanced_accuracy_score(ep_labels,  ep_preds)
        vba  = balanced_accuracy_score(val_labels, val_preds)

        history["train_loss"].append(tl)
        history["val_loss"].append(vl)
        history["train_bacc"].append(tba)
        history["val_bacc"].append(vba)

        wandb.log({"train/loss": tl, "val/loss": vl,
                   "train/balanced_acc": tba, "val/balanced_acc": vba,
                   "epoch": epoch})

        # ── checkpoint + early stopping ───────────────────────────────────────
        if vba > best_val_bacc:
            best_val_bacc    = vba
            patience_counter = 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            patience_counter += 1

        if patience_counter >= config.get("early_stopping", 20):
            log.info(f"[{model_name}] early stop @ epoch {epoch+1}  best_val_bacc={best_val_bacc:.4f}")
            break

    model.load_state_dict(torch.load(ckpt_path, weights_only=True))
    run.summary["best_val_bacc"] = best_val_bacc
    run.summary["train_time_s"]  = round(time.time() - t_start, 1)
    run.finish()

    log.info(f"[{model_name}] best_val_bacc={best_val_bacc:.4f}  "
             f"time={time.time()-t_start:.0f}s")
    return model, history


def evaluate_model(
    model: nn.Module,
    test_loader,
    config: dict,
    model_name: str,
    is_dann: bool = False,
) -> dict:
    """Evaluate on test set. Returns metrics dict."""
    device = torch.device(config["device"])
    model  = model.to(device).eval()

    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in test_loader:
            batch  = batch.to(device)
            logits = model.predict(batch) if is_dann else model(batch)
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_labels.extend(batch.y.cpu().numpy())

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)

    bacc         = balanced_accuracy_score(all_labels, all_preds)
    macro_f1     = f1_score(all_labels, all_preds, average="macro",  zero_division=0)
    per_cls_f1   = f1_score(all_labels, all_preds, average=None,     zero_division=0)
    cm           = confusion_matrix(all_labels, all_preds,
                                    labels=list(range(config["n_classes"])))

    log.info(f"[{model_name}] test_bacc={bacc:.4f}  macro_f1={macro_f1:.4f}")
    return {
        "model":            model_name,
        "test_bacc":        bacc,
        "macro_f1":         macro_f1,
        "per_class_f1":     per_cls_f1,
        "confusion_matrix": cm,
        "preds":            all_preds,
        "labels":           all_labels,
    }


log.info("Training & evaluation functions OK.")

01:30:41  INFO      Training & evaluation functions OK.


## 6 — Train All Models

In [9]:
# ── Model A: GCN ─────────────────────────────────────────────────────────────
gcn_model = GCNModel(
    in_channels = IN_CHANNELS,
    hidden_dim  = CONFIG["hidden_dim"],
    n_classes   = CONFIG["n_classes"],
    dropout     = CONFIG["dropout"],
)
log.info(f"GCN  params: {n_params(gcn_model):,}")

t0 = time.time()
gcn_model, gcn_history = train_model(
    gcn_model, train_loader, val_loader, CONFIG,
    model_name="GCN", class_weights=_loss_weights,
)
gcn_time = time.time() - t0

# ── Model B: GAT ─────────────────────────────────────────────────────────────
gat_model = GATModel(
    in_channels = IN_CHANNELS,
    hidden_dim  = CONFIG["hidden_dim"],
    n_classes   = CONFIG["n_classes"],
    heads       = CONFIG["gat_heads"],
    dropout     = CONFIG["dropout"],
)
log.info(f"GAT  params: {n_params(gat_model):,}")

t0 = time.time()
gat_model, gat_history = train_model(
    gat_model, train_loader, val_loader, CONFIG,
    model_name="GAT", class_weights=_loss_weights,
)
gat_time = time.time() - t0

# ── Model C: DANN ─────────────────────────────────────────────────────────────
dann_model = DANNModel(
    in_channels = IN_CHANNELS,
    hidden_dim  = CONFIG["hidden_dim"],
    n_classes   = CONFIG["n_classes"],
    n_subjects  = N_SUBJECTS,
    dann_lambda = CONFIG["dann_lambda"],
    dropout     = CONFIG["dropout"],
)
log.info(f"DANN params: {n_params(dann_model):,}")

t0 = time.time()
dann_model, dann_history = train_model(
    dann_model, train_loader, val_loader, CONFIG,
    model_name="DANN", class_weights=_loss_weights, is_dann=True,
)
dann_time = time.time() - t0

01:30:47  INFO      GCN  params: 35,172
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


wandb: Initializing weave.


Output()

01:30:49  INFO      HTTP Request: GET https://trace.wandb.ai/server_info "HTTP/1.1 200 OK"
01:30:50  INFO      HTTP Request: POST https://api.wandb.ai/graphql "HTTP/1.1 200 OK"
01:30:50  INFO      HTTP Request: POST https://api.wandb.ai/graphql "HTTP/1.1 200 OK"
01:30:50  INFO      HTTP Request: GET https://trace.wandb.ai/server_info "HTTP/1.1 200 OK"
01:30:50  INFO      HTTP Request: GET https://pypi.org/pypi/wandb/json "HTTP/1.1 200 OK"
01:30:50  INFO      HTTP Request: GET https://pypi.org/pypi/weave/json "HTTP/1.1 200 OK"
weave: Logged in as Weights & Biases user: uras-daniele22.
weave: View Weave data at https://wandb.ai/uras-daniele22-politecnico-di-milano/miralis-imagined-speech/weave
01:30:50  INFO      Logged in as Weights & Biases user: uras-daniele22.
View Weave data at https://wandb.ai/uras-daniele22-politecnico-di-milano/miralis-imagined-speech/weave


[GCN]:   0%|          | 0/100 [00:00<?, ?it/s]

...

01:37:21  INFO      HTTP Request: GET https://trace.wandb.ai/server_info "HTTP/1.1 200 OK"
01:37:21  INFO      HTTP Request: POST https://api.wandb.ai/graphql "HTTP/1.1 200 OK"
01:37:21  INFO      HTTP Request: POST https://api.wandb.ai/graphql "HTTP/1.1 200 OK"
01:37:21  INFO      HTTP Request: GET https://trace.wandb.ai/server_info "HTTP/1.1 200 OK"
01:37:21  INFO      HTTP Request: GET https://pypi.org/pypi/wandb/json "HTTP/1.1 200 OK"
01:37:21  INFO      HTTP Request: GET https://pypi.org/pypi/weave/json "HTTP/1.1 200 OK"
weave: Logged in as Weights & Biases user: uras-daniele22.
weave: View Weave data at https://wandb.ai/uras-daniele22-politecnico-di-milano/miralis-imagined-speech/weave
01:37:21  INFO      Logged in as Weights & Biases user: uras-daniele22.
View Weave data at https://wandb.ai/uras-daniele22-politecnico-di-milano/miralis-imagined-speech/weave


[GAT]:   0%|          | 0/100 [00:00<?, ?it/s]

KeyboardInterrupt: 

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7561d4220b10>> (for post_run_cell), with arguments args (<ExecutionResult object at 7561d46bbfd0, execution_count=9 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 7561d46b8e10, raw_cell="# ── Model A: GCN ────────────────────────────────.." transformed_cell="# ── Model A: GCN ────────────────────────────────.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2B7b22686f73744e616d65223a227370696e6c6162733031227d/home/daniele_u/miralis-hypergraph-imagined-speech/notebooks/EEG_08_gnn_classification.ipynb#X44sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost

## 7 — Evaluation & Comparison

In [ ]:
# ── evaluate ─────────────────────────────────────────────────────────────────
results = {}
for name, model, is_dann, tt in [
    ("GCN",  gcn_model,  False, gcn_time),
    ("GAT",  gat_model,  False, gat_time),
    ("DANN", dann_model, True,  dann_time),
]:
    res = evaluate_model(model, test_loader, CONFIG, name, is_dann=is_dann)
    res["train_time_s"] = round(tt, 1)
    res["n_params"]     = n_params(model)
    results[name]       = res

# ── confusion matrices ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, res) in zip(axes, results.items()):
    sns.heatmap(
        res["confusion_matrix"], annot=True, fmt="d", cmap="Blues",
        xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES,
        ax=ax, cbar=False,
    )
    ax.set_title(f"{name}  (bAcc={res['test_bacc']:.3f})", fontweight="bold")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
fig.suptitle("Confusion Matrices — Test Set", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(project_root / "figures" / "eeg08_confusion_matrices.png", dpi=150)
plt.show()

# ── learning curves ───────────────────────────────────────────────────────────
_hist = {"GCN": gcn_history, "GAT": gat_history, "DANN": dann_history}
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for name, h in _hist.items():
    axes[0].plot(h["train_loss"], label=f"{name} train", ls="-")
    axes[0].plot(h["val_loss"],   label=f"{name} val",   ls="--")
    axes[1].plot(h["train_bacc"], label=f"{name} train", ls="-")
    axes[1].plot(h["val_bacc"],   label=f"{name} val",   ls="--")
for ax, yl, tl in zip(axes,
        ["Loss", "Balanced Accuracy"],
        ["Loss curves", "Balanced accuracy curves"]):
    ax.set_xlabel("Epoch"); ax.set_ylabel(yl)
    ax.set_title(tl); ax.legend(fontsize=7); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(project_root / "figures" / "eeg08_learning_curves.png", dpi=150)
plt.show()

# ── log test metrics to W&B (one run per model) ───────────────────────────────
for name, res in results.items():
    run = wandb.init(
        project=CONFIG["wandb_project"],
        entity =CONFIG.get("wandb_entity"),
        name   =f"{name}_test",
        reinit ="allow",
    )
    run.summary["test_bacc"] = res["test_bacc"]
    run.summary["macro_f1"]  = res["macro_f1"]
    for i, f1 in enumerate(res["per_class_f1"]):
        run.summary[f"f1_{LABEL_NAMES[i]}"] = float(f1)
    run.log({"confusion_matrix": wandb.plot.confusion_matrix(
        probs=None,
        y_true=res["labels"].tolist(),
        preds=res["preds"].tolist(),
        class_names=LABEL_NAMES,
    )})
    run.finish()

In [ ]:
# ── summary comparison table ──────────────────────────────────────────────────
rows = []
for name, res in results.items():
    row = {
        "Model":             name,
        "Test Balanced Acc": round(res["test_bacc"], 4),
        "Macro F1":          round(res["macro_f1"],  4),
        **{f"F1 {LABEL_NAMES[i]}": round(float(res["per_class_f1"][i]), 4)
           for i in range(CONFIG["n_classes"])},
        "Params":            res["n_params"],
        "Train time (s)":    res["train_time_s"],
    }
    rows.append(row)

summary_df = pd.DataFrame(rows).set_index("Model")
display(summary_df)

# log to W&B as a Table
_run = wandb.init(
    project=CONFIG["wandb_project"],
    entity =CONFIG.get("wandb_entity"),
    name   ="summary",
    reinit ="allow",
)
_run.log({"model_comparison": wandb.Table(dataframe=summary_df.reset_index())})
_run.finish()

# log to Weave as a Dataset artifact
_weave_ds = weave.Dataset(
    name="eeg08_model_comparison",
    rows=summary_df.reset_index().to_dict("records"),
)
weave.publish(_weave_ds)
log.info("Summary logged to W&B and Weave.")

---
## Assumptions

### `edge_attr` dimensionality
- Expected: `[E]` (scalar weight per edge, e.g. abs PCC value) or `[E, F]` (multi-dim).
- For **GCNConv**: the first dimension is used as `edge_weight` (per-edge scalar multiplier in the normalised Laplacian).
- For **GATConv**: edge_attr is not passed (attention computed from node features only). Adding edge features to attention would require `GATConv(edge_dim=...)` and a code change.

### Node feature shape `x`
- Expected: `[N_electrodes, N_features]` where `N_electrodes = 61` and `N_features = 384` (raw z-scored EEG time samples).
- `in_channels` is auto-detected from the first `.pt` file (`x.shape[1]`).
- If `x` contains extracted spectral/temporal features instead of raw samples, the architecture is unchanged; only `in_channels` differs.

### `subject_id` in `meta`
- Expected: `meta["subject_id"]` is an integer (0-indexed or arbitrary).
- Raw subject IDs are mapped to consecutive integers `[0, N_subjects)` for the DANN domain classifier.
- Subject IDs are parsed from the folder name `P{XXX}_S{YYY}` as a fallback if `meta` is missing.

### Label `y`
- Expected: integer in `[0, 3]` — four-class `concr4` scheme (CONCR / AZIONE / STATO / ASTRATTO).
- If stored as a 1-element tensor, it is squeezed to scalar.

### Directory structure
```
data/graphs_abs_pcc/
    P000_S001/
        trial_000.pt
        trial_001.pt
        ...
    P000_S002/
    ...
```